# Binary test

This notebook aims at presenting our work on binary classification for geolocation. The underlying goal is to be able to determine if the location assigned to an image is true or not using diffusion-based models. 

## SET-UP

### 0) Imports and utilitaries

In [ ]:
from pathlib import Path
from typing import Any

from PIL import Image
import torch
from torch.utils.data import  Dataset, DataLoader, Subset
from torchvision import transforms
from tqdm import trange, tqdm
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
import math
import random
import os
from huggingface_hub import hf_hub_download
import zipfile
import pandas as pd
import gc

from plonk.data.data import Baseline
from plonk.pipe import PlonkPipeline

In [ ]:
def random_point(lat0, lon0, radius_km):
    R_earth = 6371  # km

    # Convertir en radians
    lat0_rad = math.radians(lat0)
    lon0_rad = math.radians(lon0)

    # Distance aléatoire uniforme dans le disque
    u = random.random()
    r = radius_km * math.sqrt(u)

    # Angle aléatoire
    theta = random.uniform(0, 2*math.pi)

    # Formules de géodésie
    lat_rad = math.asin(math.sin(lat0_rad) * math.cos(r/R_earth) + 
                        math.cos(lat0_rad) * math.sin(r/R_earth) * math.cos(theta))
    lon_rad = lon0_rad + math.atan2(math.sin(theta) * math.sin(r/R_earth) * math.cos(lat0_rad),
                                    math.cos(r/R_earth) - math.sin(lat0_rad) * math.sin(lat_rad))

    # Retour en degrés
    return math.degrees(lat_rad), math.degrees(lon_rad)

### I) Data preparation

First we have to set-up the datasets for them to be used properly.

We used 2 datasets in this project : YFCC100M and osv5m

- YFCC100M is a datset with general purpose images

- osv5m is a dataset with images representing mostly landscapes (roads)

You can find below the code to prepare properly those datasets

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, transform=None):
        self.metadata=[]
        if transform is None:
            self.transform = transforms.Compose([
                transforms.Resize((336, 336)),
            ])
        else:
            self.transform = transform


    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        item = self.metadata[idx]
        #img = Image.open(item["img_path"]).convert("RGB")
        #img = self.transform(img)
        # Retourne l'image + métadonnées si besoin
        return {
            "image": str(item["img_path"]),
            "lat": item["lat"],
            "lon": item["lon"],
            "index": idx

        }

    def compute_likelihood(self, pipeline):
        dataloader = DataLoader(self, batch_size=32,num_workers=0, shuffle=False)
        
        for batch in tqdm(dataloader, desc="Processing true samples"):

            images_paths = batch["image"]
            lats = batch["lat"]
            lons = batch["lon"]
            indices= batch["index"]

            targets = torch.stack([lats, lons], dim=1)
            images=[self.transform(Image.open(image_path).convert("RGB")) for image_path in images_paths]
            with torch.no_grad():
                batch_likelihoods = pipeline.compute_likelihood(images=images, coordinates=targets)

            batch_likelihoods = batch_likelihoods.detach().cpu().numpy()

            for i, global_idx in enumerate(indices):
                self.metadata[global_idx.item()]["likelihood"] = batch_likelihoods[i]

    def compute_false_likelihood_shuffle(self, pipeline):
        # We compute false likelihood by computing the likelyhood of the location of another image 
        dataloader = DataLoader(self, batch_size=32,num_workers=0, shuffle=False)
        
        for batch in tqdm(dataloader, desc="Processing true samples"):
            images_paths = batch["image"]
            lats = batch["lat"]
            lons = batch["lon"]
            false_lats = torch.roll(lats, shifts=-1)
            false_lons = torch.roll(lons, shifts=-1)
            indices= batch["index"]


            targets = torch.stack([false_lats, false_lons], dim=1)
            images=[self.transform(Image.open(image_path).convert("RGB")) for image_path in images_paths]
            with torch.no_grad():
                batch_likelihoods = pipeline.compute_likelihood(images=images, coordinates=targets)

            batch_likelihoods = batch_likelihoods.detach().cpu().numpy()

            for i, global_idx in enumerate(indices):
                self.metadata[global_idx.item()]["false_likelihood_shuffle"] = batch_likelihoods[i]
                self.metadata[global_idx.item()]["false_lat_shuffle"] = false_lats[i].item()
                self.metadata[global_idx.item()]["false_lon_shuffle"] = false_lons[i].item()

    
    
    def compute_false_likelihood_distance(self, pipeline, radius_km):
        # We compute false likelihood by computing the likelyhood of the location of another image 
        dataloader = DataLoader(self, batch_size=32,num_workers=0, shuffle=False)

        for batch in tqdm(dataloader, desc="Processing true samples"):
            images_paths = batch["image"]
            lats = batch["lat"]
            lons = batch["lon"]
            indices= batch["index"]

            false_lats = []
            false_lons = []
            
            for lat0, lon0 in zip(lats, lons):
                lat_noisy, lon_noisy = random_point(lat0, lon0, radius_km)
                false_lats.append(lat_noisy)
                false_lons.append(lon_noisy)



            targets = torch.stack([torch.tensor(false_lats), torch.tensor(false_lons)], dim=1)
            images=[self.transform(Image.open(image_path).convert("RGB")) for image_path in images_paths]
            with torch.no_grad():
                batch_likelihoods = pipeline.compute_likelihood(images=images, coordinates=targets)

            batch_likelihoods = batch_likelihoods.detach().cpu().numpy()

            for i, global_idx in enumerate(indices):
                self.metadata[global_idx.item()][f"false_likelihood_{radius_km}"] = batch_likelihoods[i]
                self.metadata[global_idx.item()][f"false_lat_{radius_km}"] = false_lats[i]
                self.metadata[global_idx.item()][f"false_lon_{radius_km}"] = false_lons[i]

    def compute_avg_likelihood(self, pipeline, n):
        #the localizability computation is exactly what we want:
        #   an average likelyhood among probable locations 
        for i in trange(min(n, len(self.metadata))):
            item = self.metadata[i]
            image_path=item["img_path"]
            image=self.transform(Image.open(image_path).convert("RGB"))
            localizability= pipeline.compute_localizability(image, number_monte_carlo_samples=10)
            item["avg_likelihood"]= localizability*(4 * torch.pi * np.log(2)) #renormalisation





##### I.1. YFCC100M

In [ ]:
class YFCC4kCustomDataset(CustomDataset):
    def __init__(self, dataset_root, transform=None):
        """
        dataset_root: chemin vers le dossier contenant 'images' et 'info.txt'
        transform: transformations PyTorch (optionnel)
        """
        super().__init__(transform=transform)
        self.dataset_root = Path(dataset_root)
        self.image_dir = self.dataset_root / "images"

        with open(self.dataset_root / "info.txt", "r") as f:
            for line in f:
                #In this metadata file, each line maps to one image, and the format is the following:
                #           img_num \t lon \t lat
                parts = line.strip().split("\t")
                img_num = parts[0]
                lon = float(parts[-2])
                lat = float(parts[-1])

                img_path = self.image_dir / f"{img_num}.jpg"

                if img_path.exists():
                    self.metadata.append({
                        "img_path": img_path,
                        "lat": float(lat),
                        "lon": float(lon)
                    })

    def save(self, save_dir="saved_metadata" ):
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, "metadata_YFCC.pt")

        torch.save(self.metadata, save_path)

##### I.2. osv5m

In [ ]:
#not ready
class osv5mCustomDataset(CustomDataset):
    def __init__(self, dataset_root, transform=None):
        """
        dataset_root: chemin vers le dossier contenant 'images' et 'info.txt'
        transform: transformations PyTorch (optionnel)
        """
        super().__init__(transform=transform)
        self.dataset_root = Path(dataset_root)
        self.image_dir = self.dataset_root / "images/test"

        self.metadata = []

        #We only use the first 1000 images for computation-time reason.
        df = pd.read_csv(self.dataset_root /"test.csv", usecols=["id", "latitude", "longitude"], nrows=1000)
        for row in df.itertuples(index=False):

            img_num = row.id
            lon = row.longitude
            lat = row.latitude


            #We have to use following code to find the .png in one of the 5 sub-directories
            img_path = None
            for subdir in range(5):  # 00 à 04
                candidate = self.image_dir / f"{subdir:02d}" / f"{img_num}.jpg"
                if candidate.exists():
                    img_path = candidate
                    break
            
            if img_path is None:
                print(f"Image {img_num}.jpg non trouvée !")
            else:
                self.metadata.append({
                    "img_path": img_path,
                    "lat": float(lat),
                    "lon": float(lon)
                })

    def save(self, save_dir="saved_metadata" ):
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, "metadata_osv5m.pt")

        torch.save(self.metadata, save_path)


### II) Load

We use torch.save to store results of precedent computation

In [ ]:
def load_YFCC(dataset_root, save_dir = "saved_metadata", transform=None):
    #Extraction of metadata
    save_path = os.path.join(save_dir, "metadata_YFCC.pt")
    if not os.path.exists(save_path):
        raise FileNotFoundError(f"No precomputed metadata found at {save_path}. Please run dataset.save() first.")

    loaded_metadata = torch.load(save_path)
    
    #Re-creation of the dataset
    dataset = YFCC4kCustomDataset.__new__(YFCC4kCustomDataset)
    CustomDataset.__init__(dataset, transform=transform)
    dataset.metadata = loaded_metadata
    dataset.dataset_root = Path(dataset_root)
    dataset.image_dir = dataset.dataset_root / "images"
    return dataset

def load_osv5m(dataset_root, save_dir = "saved_metadata", transform=None):
    #Extraction of metadata
    save_path = os.path.join(save_dir, "metadata_osv5m.pt")
    if not os.path.exists(save_path):
        raise FileNotFoundError(f"No precomputed metadata found at {save_path}. Please run dataset.save() first.")

    loaded_metadata = torch.load(save_path)
    
    #Re-creation of the dataset
    dataset = osv5mCustomDataset.__new__(osv5mCustomDataset)
    CustomDataset.__init__(dataset, transform=transform)
    dataset.metadata = loaded_metadata
    dataset.dataset_root = Path(dataset_root)
    dataset.image_dir = dataset.dataset_root / "images"
    return dataset

### III) Actual computation

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

pipeline= PlonkPipeline("nicolas-dufour/PLONK_iNaturalist").to(device)

In [ ]:
try:
    YFCC_ds = load_YFCC(dataset_root = "../../datasets/YFCC100M/yfcc4k", transform =None)
except:
    YFCC_ds = YFCC4kCustomDataset(dataset_root = "../../datasets/YFCC100M/yfcc4k", transform =None)

if "likelihood" not in YFCC_ds.metadata[0]:
    print("Computing likelihood")
    gc.collect()
    torch.cuda.empty_cache()
    YFCC_ds.compute_likelihood(pipeline)
    YFCC_ds.save()

if "false_likelihood_shuffle" not in YFCC_ds.metadata[0]:
    print("Computing false_likelihood (shuffle)")
    gc.collect()
    torch.cuda.empty_cache()
    YFCC_ds.compute_false_likelihood_shuffle(pipeline)
    YFCC_ds.save()

if "false_likelihood_100" not in YFCC_ds.metadata[0]:
    print("Computing false_likelihood (100)")
    gc.collect()
    torch.cuda.empty_cache()
    YFCC_ds.compute_false_likelihood_distance(pipeline, 100)
    YFCC_ds.save()

if "false_likelihood_1000" not in YFCC_ds.metadata[0]:
    print("Computing false_likelihood (1000)")
    gc.collect()
    torch.cuda.empty_cache()
    YFCC_ds.compute_false_likelihood_distance(pipeline, 1000)
    YFCC_ds.save()

if "avg_likelihood" not in YFCC_ds.metadata[0]:
    print("Computing avg_likelihood")
    gc.collect()
    torch.cuda.empty_cache()
    YFCC_ds.compute_avg_likelihood(pipeline, 100) # We only compute avg_likelyhood for the first 100 images for running time issue
    YFCC_ds.save()


In [ ]:
try:
    osv_ds = load_osv5m(dataset_root = "../datasets/osv5m/", transform =None)
except:
    osv_ds = osv5mCustomDataset(dataset_root = "../datasets/osv5m/", transform =None)

if "likelihood" not in osv_ds.metadata[0]:
    print("Computing likelihood")
    gc.collect()
    torch.cuda.empty_cache()
    osv_ds.compute_likelihood(pipeline)
    osv_ds.save()

if "false_likelihood_shuffle" not in osv_ds.metadata[0]:
    print("Computing false_likelihood (shuffle)")
    gc.collect()
    torch.cuda.empty_cache()
    osv_ds.compute_false_likelihood_shuffle(pipeline)
    osv_ds.save()


if "false_likelihood_100" not in osv_ds.metadata[0]:
    print("Computing false_likelihood (100)")
    gc.collect()
    torch.cuda.empty_cache()
    osv_ds.compute_false_likelihood_distance(pipeline, 100)
    osv_ds.save()

if "false_likelihood_1000" not in osv_ds.metadata[0]:
    print("Computing false_likelihood (1000)")
    gc.collect()
    torch.cuda.empty_cache()
    osv_ds.compute_false_likelihood_distance(pipeline, 1000)
    osv_ds.save()

if "avg_likelihood" not in osv_ds.metadata[0]:
    print("Computing avg_likelihood")
    gc.collect()
    torch.cuda.empty_cache()
    osv_ds.compute_avg_likelihood(pipeline, 100) # We only compute avg_likelyhood for the first 100 images for running time issue
    osv_ds.save()

## First method : Only using Likelihood

We try to determine if a threshold method is viable for the binary classification.

### Previsualisation

##### With general purpose images (YFCC dataset)

In [ ]:
y_true_scores= [item["likelihood"] for item in YFCC_ds.metadata]
y_false_scores_shuffle= [item["false_likelihood_shuffle"] for item in YFCC_ds.metadata]
y_false_scores_100= [item["false_likelihood_100"] for item in YFCC_ds.metadata]
y_false_scores_1000= [item["false_likelihood_1000"] for item in YFCC_ds.metadata]


y_true_scores = np.array(y_true_scores)
y_false_scores_shuffle = np.array(y_false_scores_shuffle)
y_false_scores_100 = np.array(y_false_scores_100)
y_false_scores_1000 = np.array(y_false_scores_1000)

bins = np.linspace(
    min(
        min(y_true_scores),
        min(y_false_scores_shuffle),
        min(y_false_scores_100),
        min(y_false_scores_1000)
    ),
    max(
        max(y_true_scores),
        max(y_false_scores_shuffle),
        max(y_false_scores_100),
        max(y_false_scores_1000)
    ),
    50
)


fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# True vs Shuffle
axs[0].hist(y_true_scores, bins=bins, alpha=0.5, label="true")
axs[0].hist(y_false_scores_shuffle, bins=bins, alpha=0.5, label="shuffle")
axs[0].set_title("True vs Shuffle")
axs[0].legend()

# True vs 100
axs[1].hist(y_true_scores, bins=bins, alpha=0.5, label="true")
axs[1].hist(y_false_scores_100, bins=bins, alpha=0.5, label="100 km")
axs[1].set_title("True vs 100 km")
axs[1].legend()

# True vs 1000
axs[2].hist(y_true_scores, bins=bins, alpha=0.5, label="true")
axs[2].hist(y_false_scores_1000, bins=bins, alpha=0.5, label="1000 km")
axs[2].set_title("True vs 1000 km")
axs[2].legend()

fig.suptitle("Distributions of the likelihoods true coordinates vs false coordinates (generated with 3 different methods) for the YFCC dataset", fontsize=16)
plt.tight_layout()
plt.show()

##### With landscapes images (osv dataset)

In [ ]:
o_true_scores= [item["likelihood"] for item in osv_ds.metadata]
o_false_scores_shuffle= [item["false_likelihood_shuffle"] for item in osv_ds.metadata]
o_false_scores_100= [item["false_likelihood_100"] for item in osv_ds.metadata]
o_false_scores_1000= [item["false_likelihood_1000"] for item in osv_ds.metadata]

o_true_scores = np.array(o_true_scores)
o_false_scores_shuffle = np.array(o_false_scores_shuffle)
o_false_scores_100 = np.array(o_false_scores_100)
o_false_scores_1000 = np.array(o_false_scores_1000)

bins = np.linspace(
    min(
        o_true_scores.min(),
        o_false_scores_shuffle.min(),
        o_false_scores_100.min(),
        o_false_scores_1000.min()
    ),
    max(
        o_true_scores.max(),
        o_false_scores_shuffle.max(),
        o_false_scores_100.max(),
        o_false_scores_1000.max()
    ),
    50
)

fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# True vs Shuffle
axs[0].hist(o_true_scores, bins=bins, alpha=0.5, label="true")
axs[0].hist(o_false_scores_shuffle, bins=bins, alpha=0.5, label="shuffle")
axs[0].set_title("True vs Shuffle")
axs[0].legend()

# True vs 100
axs[1].hist(o_true_scores, bins=bins, alpha=0.5, label="true")
axs[1].hist(o_false_scores_100, bins=bins, alpha=0.5, label="100 km")
axs[1].set_title("True vs 100 km")
axs[1].legend()

# True vs 1000
axs[2].hist(o_true_scores, bins=bins, alpha=0.5, label="true")
axs[2].hist(o_false_scores_1000, bins=bins, alpha=0.5, label="1000 km")
axs[2].set_title("True vs 1000 km")
axs[2].legend()

fig.suptitle("Distributions of the likelihoods true coordinates vs false coordinates (generated with 3 different methods) for the osv dataset", fontsize=16)
plt.tight_layout()
plt.show()

### Statistical determination of an optimal threshold

##### With general purpose images (YFCC dataset)

In [ ]:
thresholds = np.linspace(
    min(y_true_scores.min(), y_false_scores_shuffle.min(),y_false_scores_100.min(),y_false_scores_1000.min() ),
    max(y_true_scores.max(), y_false_scores_shuffle.max(),y_false_scores_100.max(), y_false_scores_1000.max()),
    200
)

y_recalls = []
y_precisions_shuffle = []
y_precisions_100 = []
y_precisions_1000 = []
y_f1s_shuffle = []
y_f1s_100 = []
y_f1s_1000 = []
y_accuracy_shuffle = []
y_accuracy_100 = []
y_accuracy_1000 = []
y_mcc_shuffle = []
y_mcc_100 = []
y_mcc_1000 = []

for t in thresholds:

    TP = (y_true_scores > t).sum().item()
    FN = (y_true_scores <= t).sum().item()

    FP_shuffle = (y_false_scores_shuffle > t).sum().item()
    TN_shuffle = (y_false_scores_shuffle <= t).sum().item()

    FP_100 = (y_false_scores_100 > t).sum().item()
    TN_100 = (y_false_scores_100 <= t).sum().item()

    FP_1000 = (y_false_scores_1000 > t).sum().item()
    TN_1000 = (y_false_scores_1000 <= t).sum().item()


    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    precision_shuffle = TP / (TP + FP_shuffle) if (TP + FP_shuffle) > 0 else 1
    precision_100 = TP / (TP + FP_100) if (TP + FP_100) > 0 else 1
    precision_1000 = TP / (TP + FP_1000) if (TP + FP_1000) > 0 else 1
    accuracy_shuffle = (TP + TN_shuffle) / (TP + FN + FP_shuffle + TN_shuffle)
    accuracy_100 = (TP + TN_100) / (TP + FN + FP_100 + TN_100)
    accuracy_1000 = (TP + TN_1000) / (TP + FN + FP_1000 + TN_1000)
    mcc_shuffle = (TP*TN_shuffle - FP_shuffle*FN) / np.sqrt((TP+FP_shuffle)*(TP+FN)*(TN_shuffle+FP_shuffle)*(TN_shuffle+FN) + 1e-10)
    mcc_100   = (TP*TN_100 - FP_100*FN) / np.sqrt((TP+FP_100)*(TP+FN)*(TN_100+FP_100)*(TN_100+FN) + 1e-10)
    mcc_1000  = (TP*TN_1000 - FP_1000*FN) / np.sqrt((TP+FP_1000)*(TP+FN)*(TN_1000+FP_1000)*(TN_1000+FN) + 1e-10)

    if precision_shuffle + recall > 0:
        f1_shuffle = 2 * precision_shuffle * recall / (precision_shuffle + recall)
    else:
        f1_shuffle = 0
    if precision_100 + recall > 0:
        f1_100 = 2 * precision_100 * recall / (precision_100 + recall)
    else:
        f1_100 = 0
    if precision_1000 + recall > 0:
        f1_1000 = 2 * precision_1000 * recall / (precision_1000 + recall)
    else:
        f1_1000 = 0

    y_recalls.append(recall)
    y_precisions_shuffle.append(precision_shuffle)
    y_precisions_100.append(precision_100)
    y_precisions_1000.append(precision_1000)
    y_f1s_shuffle.append(f1_shuffle)
    y_f1s_100.append(f1_100)
    y_f1s_1000.append(f1_1000)
    y_accuracy_shuffle.append(accuracy_shuffle)
    y_accuracy_100.append(accuracy_100)
    y_accuracy_1000.append(accuracy_1000)
    y_mcc_shuffle.append(mcc_shuffle)
    y_mcc_100.append(mcc_100)
    y_mcc_1000.append(mcc_1000)
    


In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(18, 5))

# Recalls
axs[0].plot(thresholds, y_recalls )
axs[0].set_xlabel("Threshold")
axs[0].set_ylabel("Recall")
axs[0].set_title("Recalls")
axs[0].legend()

# Precision shuffle
axs[1].plot(thresholds, y_precisions_shuffle )
axs[1].set_xlabel("Threshold")
axs[1].set_ylabel("Precision")
axs[1].set_title("Precision (shuffle)")
axs[1].legend()

# Precision 100km
axs[2].plot(thresholds, y_precisions_100)
axs[2].set_xlabel("Threshold")
axs[2].set_ylabel("Precision")
axs[2].set_title("Precision (100km)")
axs[2].legend()
# Precision 1000km
axs[3].plot(thresholds, y_precisions_1000)
axs[3].set_xlabel("Threshold")
axs[3].set_ylabel("Precision")
axs[3].set_title("Precision (1000km)")
axs[3].legend()

fig.suptitle("Recalls and Precisions given different thresholds for the YFCC dataset", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Recalls
axs[0].plot(y_recalls, y_precisions_shuffle )
axs[0].set_xlabel("Recall")
axs[0].set_ylabel("Precision")
axs[0].set_title("Precision-Recall curve (shuffle)")
axs[0].legend()

# Precision 100km
axs[1].plot(y_recalls, y_precisions_100)
axs[1].set_xlabel("Recall")
axs[1].set_ylabel("Precision")
axs[1].set_title("Precision-Recall curve (100km)")
axs[1].legend()
# Precision 1000km
axs[2].plot(y_recalls, y_precisions_1000)
axs[2].set_xlabel("Recall")
axs[2].set_ylabel("Precision")
axs[2].set_title("Precision-Recall curve(1000km)")
axs[2].legend()

fig.suptitle("Precision-Recall curve for the YFCC dataset", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Recalls
axs[0].plot(thresholds, y_f1s_shuffle )
axs[0].axhline(y=2/3, color='red', linestyle='--')
axs[0].set_xlabel("Threshold")
axs[0].set_ylabel("F1 score")
axs[0].set_title("F1 score for different thresholds (shuffle)")
axs[0].legend()

# Precision 100km
axs[1].plot(thresholds, y_f1s_100)
axs[1].axhline(y=2/3, color='red', linestyle='--')
axs[1].set_xlabel("Threshold")
axs[1].set_ylabel("F1 score")
axs[1].set_title("F1 score for different thresholds (100km)")
axs[1].legend()
# Precision 1000km
axs[2].plot(thresholds, y_f1s_1000)
axs[2].axhline(y=2/3, color='red', linestyle='--')
axs[2].set_xlabel("Threshold")
axs[2].set_ylabel("F1 score")
axs[2].set_title("F1 score for different thresholds (1000km)")
axs[2].legend()

fig.suptitle("F1 score given different thresholds for the YFCC dataset", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Recalls
axs[0].plot(thresholds, y_accuracy_shuffle )
axs[0].set_xlabel("Threshold")
axs[0].set_ylabel("Accuracy")
axs[0].set_title("Accuracy for different thresholds (shuffle)")
axs[0].legend()

# Precision 100km
axs[1].plot(thresholds, y_accuracy_100)
axs[1].set_xlabel("Threshold")
axs[1].set_ylabel("Accuracy")
axs[1].set_title("Accuracy for different thresholds (100km)")
axs[1].legend()
# Precision 1000km
axs[2].plot(thresholds, y_accuracy_1000)
axs[2].set_xlabel("Threshold")
axs[2].set_ylabel("Accuracy")
axs[2].set_title("Accuracy for different thresholds (1000km)")
axs[2].legend()

fig.suptitle("Accuracy given different thresholds for the YFCC dataset", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Recalls
axs[0].plot(thresholds, y_mcc_shuffle )
axs[0].set_xlabel("Threshold")
axs[0].set_ylabel("MCC")
axs[0].set_title("MCC for different thresholds (shuffle)")
axs[0].legend()

# Precision 100km
axs[1].plot(thresholds, y_mcc_100)
axs[1].set_xlabel("Threshold")
axs[1].set_ylabel("MCC")
axs[1].set_title("MCC for different thresholds (100km)")
axs[1].legend()
# Precision 1000km
axs[2].plot(thresholds, y_mcc_1000)
axs[2].set_xlabel("Threshold")
axs[2].set_ylabel("MCC")
axs[2].set_title("MCC for different thresholds (1000km)")
axs[2].legend()

fig.suptitle("Matthews Correlation Coefficient given different thresholds for the YFCC dataset", fontsize=16)
plt.tight_layout()
plt.show()

##### With landscapes images (osv dataset)

In [ ]:
thresholds = np.linspace(
    min(o_true_scores.min(), o_false_scores_shuffle.min(),o_false_scores_100.min(),o_false_scores_1000.min() ),
    max(o_true_scores.max(), o_false_scores_shuffle.max(),o_false_scores_100.max(), o_false_scores_1000.max()),
    200
)

o_recalls = []
o_precisions_shuffle = []
o_precisions_100 = []
o_precisions_1000 = []
o_f1s_shuffle = []
o_f1s_100 = []
o_f1s_1000 = []
o_accuracy_shuffle = []
o_accuracy_100 = []
o_accuracy_1000 = []
o_mcc_shuffle = []
o_mcc_100 = []
o_mcc_1000 = []



for t in thresholds:

    TP = (o_true_scores > t).sum().item()
    FN = (o_true_scores <= t).sum().item()

    FP_shuffle = (o_false_scores_shuffle > t).sum().item()
    TN_shuffle = (o_false_scores_shuffle <= t).sum().item()

    FP_100 = (o_false_scores_100 > t).sum().item()
    TN_100 = (o_false_scores_100 <= t).sum().item()

    FP_1000 = (o_false_scores_1000 > t).sum().item()
    TN_1000 = (o_false_scores_1000 <= t).sum().item()


    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    precision_shuffle = TP / (TP + FP_shuffle) if (TP + FP_shuffle) > 0 else 1
    precision_100 = TP / (TP + FP_100) if (TP + FP_100) > 0 else 1
    precision_1000 = TP / (TP + FP_1000) if (TP + FP_1000) > 0 else 1
    accuracy_shuffle = (TP + TN_shuffle) / (TP + FN + FP_shuffle + TN_shuffle)
    accuracy_100 = (TP + TN_100) / (TP + FN + FP_100 + TN_100)
    accuracy_1000 = (TP + TN_1000) / (TP + FN + FP_1000 + TN_1000)
    mcc_shuffle = (TP*TN_shuffle - FP_shuffle*FN) / np.sqrt((TP+FP_shuffle)*(TP+FN)*(TN_shuffle+FP_shuffle)*(TN_shuffle+FN) + 1e-10)
    mcc_100   = (TP*TN_100 - FP_100*FN) / np.sqrt((TP+FP_100)*(TP+FN)*(TN_100+FP_100)*(TN_100+FN) + 1e-10)
    mcc_1000  = (TP*TN_1000 - FP_1000*FN) / np.sqrt((TP+FP_1000)*(TP+FN)*(TN_1000+FP_1000)*(TN_1000+FN) + 1e-10)


    if precision_shuffle + recall > 0:
        f1_shuffle = 2 * precision_shuffle * recall / (precision_shuffle + recall)
    else:
        f1_shuffle = 0
    if precision_100 + recall > 0:
        f1_100 = 2 * precision_100 * recall / (precision_100 + recall)
    else:
        f1_100 = 0
    if precision_1000 + recall > 0:
        f1_1000 = 2 * precision_1000 * recall / (precision_1000 + recall)
    else:
        f1_1000 = 0

    o_recalls.append(recall)
    o_precisions_shuffle.append(precision_shuffle)
    o_precisions_100.append(precision_100)
    o_precisions_1000.append(precision_1000)
    o_f1s_shuffle.append(f1_shuffle)
    o_f1s_100.append(f1_100)
    o_f1s_1000.append(f1_1000)
    o_accuracy_shuffle.append(accuracy_shuffle)
    o_accuracy_100.append(accuracy_100)
    o_accuracy_1000.append(accuracy_1000)
    o_mcc_shuffle.append(mcc_shuffle)
    o_mcc_100.append(mcc_100)
    o_mcc_1000.append(mcc_1000)


In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(18, 5))

# Recalls
axs[0].plot(thresholds, o_recalls )
axs[0].set_xlabel("Threshold")
axs[0].set_ylabel("Recall")
axs[0].set_title("Recalls")
axs[0].legend()

# Precision shuffle
axs[1].plot(thresholds, o_precisions_shuffle )
axs[1].set_xlabel("Threshold")
axs[1].set_ylabel("Precision")
axs[1].set_title("Precision (shuffle)")
axs[1].legend()

# Precision 100km
axs[2].plot(thresholds, o_precisions_100)
axs[2].set_xlabel("Threshold")
axs[2].set_ylabel("Precision")
axs[2].set_title("Precision (100km)")
axs[2].legend()
# Precision 1000km
axs[3].plot(thresholds, o_precisions_1000)
axs[3].set_xlabel("Threshold")
axs[3].set_ylabel("Precision")
axs[3].set_title("Precision (1000km)")
axs[3].legend()

fig.suptitle("Recalls and Precisions given different thresholds for the OSV dataset", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Recalls
axs[0].plot(o_recalls, o_precisions_shuffle )
axs[0].set_xlabel("Recall")
axs[0].set_ylabel("Precision")
axs[0].set_title("Precision-Recall curve (shuffle)")
axs[0].legend()

# Precision 100km
axs[1].plot(o_recalls, o_precisions_100)
axs[1].set_xlabel("Recall")
axs[1].set_ylabel("Precision")
axs[1].set_title("Precision-Recall curve (100km)")
axs[1].legend()
# Precision 1000km
axs[2].plot(o_recalls, o_precisions_1000)
axs[2].set_xlabel("Recall")
axs[2].set_ylabel("Precision")
axs[2].set_title("Precision-Recall curve(1000km)")
axs[2].legend()

fig.suptitle("Precision-Recall curve for the OSV dataset", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Recalls
axs[0].plot(thresholds, o_f1s_shuffle )
axs[0].axhline(y=2/3, color='red', linestyle='--')
axs[0].set_xlabel("Threshold")
axs[0].set_ylabel("F1 score")
axs[0].set_title("F1 score for different thresholds (shuffle)")
axs[0].legend()

# Precision 100km
axs[1].plot(thresholds, o_f1s_100)
axs[1].axhline(y=2/3, color='red', linestyle='--')
axs[1].set_xlabel("Threshold")
axs[1].set_ylabel("F1 score")
axs[1].set_title("F1 score for different thresholds (100km)")
axs[1].legend()
# Precision 1000km
axs[2].plot(thresholds, o_f1s_1000)
axs[2].axhline(y=2/3, color='red', linestyle='--')
axs[2].set_xlabel("Threshold")
axs[2].set_ylabel("F1 score")
axs[2].set_title("F1 score for different thresholds (1000km)")
axs[2].legend()

fig.suptitle("F1 score given different thresholds for the OSV dataset", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Recalls
axs[0].plot(thresholds, o_accuracy_shuffle )
axs[0].set_xlabel("Threshold")
axs[0].set_ylabel("Accuracy")
axs[0].set_title("Accuracy for different thresholds (shuffle)")
axs[0].legend()

# Precision 100km
axs[1].plot(thresholds, o_accuracy_100)
axs[1].set_xlabel("Threshold")
axs[1].set_ylabel("Accuracy")
axs[1].set_title("Accuracy for different thresholds (100km)")
axs[1].legend()
# Precision 1000km
axs[2].plot(thresholds, o_accuracy_1000)
axs[2].set_xlabel("Threshold")
axs[2].set_ylabel("Accuracy")
axs[2].set_title("Accuracy for different thresholds (1000km)")
axs[2].legend()

fig.suptitle("Accuracy given different thresholds for the OSV dataset", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Recalls
axs[0].plot(thresholds, o_mcc_shuffle )
axs[0].set_xlabel("Threshold")
axs[0].set_ylabel("MCC")
axs[0].set_title("MCC for different thresholds (shuffle)")
axs[0].legend()

# Precision 100km
axs[1].plot(thresholds, o_mcc_100)
axs[1].set_xlabel("Threshold")
axs[1].set_ylabel("MCC")
axs[1].set_title("MCC for different thresholds (100km)")
axs[1].legend()
# Precision 1000km
axs[2].plot(thresholds, o_mcc_1000)
axs[2].set_xlabel("Threshold")
axs[2].set_ylabel("MCC")
axs[2].set_title("MCC for different thresholds (1000km)")
axs[2].legend()

fig.suptitle("Matthews Correlation Coefficient given different thresholds for the OSV dataset", fontsize=16)
plt.tight_layout()
plt.show()

## Second method : Also using the average likelihood

### Previsualisation

##### With general purpose images (YFCC dataset)

In [ ]:
y_true_scores= [item["likelihood"] - item["avg_likelihood"].item() for item in YFCC_ds.metadata[:100]]
y_false_scores_shuffle= [item["false_likelihood_shuffle"] - item["avg_likelihood"].item() for item in YFCC_ds.metadata[:100]]
y_false_scores_100= [item["false_likelihood_100"] - item["avg_likelihood"].item() for item in YFCC_ds.metadata[:100]]
y_false_scores_1000= [item["false_likelihood_1000"] - item["avg_likelihood"].item() for item in YFCC_ds.metadata[:100]]


y_true_scores = np.array(y_true_scores)
y_false_scores_shuffle = np.array(y_false_scores_shuffle)
y_false_scores_100 = np.array(y_false_scores_100)
y_false_scores_1000 = np.array(y_false_scores_1000)

bins = np.linspace(
    min(
        min(y_true_scores),
        min(y_false_scores_shuffle),
        min(y_false_scores_100),
        min(y_false_scores_1000)
    ),
    max(
        max(y_true_scores),
        max(y_false_scores_shuffle),
        max(y_false_scores_100),
        max(y_false_scores_1000)
    ),
    50
)


fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# True vs Shuffle
axs[0].hist(y_true_scores, bins=bins, alpha=0.5, label="true")
axs[0].hist(y_false_scores_shuffle, bins=bins, alpha=0.5, label="shuffle")
axs[0].set_title("True vs Shuffle")
axs[0].set_xlabel("Delta")
axs[0].legend()

# True vs 100
axs[1].hist(y_true_scores, bins=bins, alpha=0.5, label="true")
axs[1].hist(y_false_scores_100, bins=bins, alpha=0.5, label="100 km")
axs[1].set_title("True vs 100 km")
axs[1].set_xlabel("Delta")
axs[1].legend()

# True vs 1000
axs[2].hist(y_true_scores, bins=bins, alpha=0.5, label="true")
axs[2].hist(y_false_scores_1000, bins=bins, alpha=0.5, label="1000 km")
axs[2].set_title("True vs 1000 km")
axs[2].set_xlabel("Delta")
axs[2].legend()

fig.suptitle("Distributions of the delta for the true coordinates vs false coordinates (generated with 3 different methods) for the YFCC dataset", fontsize=16)
plt.tight_layout()
plt.show()

##### With landscapes images (osv dataset)

In [ ]:
o_true_scores= [item["likelihood"]- item["avg_likelihood"].item() for item in osv_ds.metadata[:100]]
o_false_scores_shuffle= [item["false_likelihood_shuffle"]- item["avg_likelihood"].item() for item in osv_ds.metadata[:100]]
o_false_scores_100= [item["false_likelihood_100"]- item["avg_likelihood"].item() for item in osv_ds.metadata[:100]]
o_false_scores_1000= [item["false_likelihood_1000"]- item["avg_likelihood"].item() for item in osv_ds.metadata[:100]]

o_true_scores = np.array(o_true_scores)
o_false_scores_shuffle = np.array(o_false_scores_shuffle)
o_false_scores_100 = np.array(o_false_scores_100)
o_false_scores_1000 = np.array(o_false_scores_1000)

bins = np.linspace(
    min(
        o_true_scores.min(),
        o_false_scores_shuffle.min(),
        o_false_scores_100.min(),
        o_false_scores_1000.min()
    ),
    max(
        o_true_scores.max(),
        o_false_scores_shuffle.max(),
        o_false_scores_100.max(),
        o_false_scores_1000.max()
    ),
    50
)

fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# True vs Shuffle
axs[0].hist(o_true_scores, bins=bins, alpha=0.5, label="true")
axs[0].hist(o_false_scores_shuffle, bins=bins, alpha=0.5, label="shuffle")
axs[0].set_title("True vs Shuffle")
axs[0].set_xlabel("Delta")
axs[0].legend()

# True vs 100
axs[1].hist(o_true_scores, bins=bins, alpha=0.5, label="true")
axs[1].hist(o_false_scores_100, bins=bins, alpha=0.5, label="100 km")
axs[1].set_title("True vs 100 km")
axs[1].set_xlabel("Delta")
axs[1].legend()

# True vs 1000
axs[2].hist(o_true_scores, bins=bins, alpha=0.5, label="true")
axs[2].hist(o_false_scores_1000, bins=bins, alpha=0.5, label="1000 km")
axs[2].set_title("True vs 1000 km")
axs[2].set_xlabel("Delta")
axs[2].legend()

fig.suptitle("Distributions of the delta for the true coordinates vs false coordinates (generated with 3 different methods) for the osv dataset", fontsize=16)
plt.tight_layout()
plt.show()

### Statistical determination of an optimal threshold

##### With general purpose images (YFCC dataset)

In [ ]:
thresholds = np.linspace(
    min(y_true_scores.min(), y_false_scores_shuffle.min(),y_false_scores_100.min(),y_false_scores_1000.min() ),
    max(y_true_scores.max(), y_false_scores_shuffle.max(),y_false_scores_100.max(), y_false_scores_1000.max()),
    200
)

y_recalls = []
y_precisions_shuffle = []
y_precisions_100 = []
y_precisions_1000 = []
y_f1s_shuffle = []
y_f1s_100 = []
y_f1s_1000 = []
y_accuracy_shuffle = []
y_accuracy_100 = []
y_accuracy_1000 = []
y_mcc_shuffle = []
y_mcc_100 = []
y_mcc_1000 = []

for t in thresholds:

    TP = (y_true_scores > t).sum().item()
    FN = (y_true_scores <= t).sum().item()

    FP_shuffle = (y_false_scores_shuffle > t).sum().item()
    TN_shuffle = (y_false_scores_shuffle <= t).sum().item()

    FP_100 = (y_false_scores_100 > t).sum().item()
    TN_100 = (y_false_scores_100 <= t).sum().item()

    FP_1000 = (y_false_scores_1000 > t).sum().item()
    TN_1000 = (y_false_scores_1000 <= t).sum().item()


    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    precision_shuffle = TP / (TP + FP_shuffle) if (TP + FP_shuffle) > 0 else 1
    precision_100 = TP / (TP + FP_100) if (TP + FP_100) > 0 else 1
    precision_1000 = TP / (TP + FP_1000) if (TP + FP_1000) > 0 else 1
    accuracy_shuffle = (TP + TN_shuffle) / (TP + FN + FP_shuffle + TN_shuffle)
    accuracy_100 = (TP + TN_100) / (TP + FN + FP_100 + TN_100)
    accuracy_1000 = (TP + TN_1000) / (TP + FN + FP_1000 + TN_1000)
    mcc_shuffle = (TP*TN_shuffle - FP_shuffle*FN) / np.sqrt((TP+FP_shuffle)*(TP+FN)*(TN_shuffle+FP_shuffle)*(TN_shuffle+FN) + 1e-10)
    mcc_100   = (TP*TN_100 - FP_100*FN) / np.sqrt((TP+FP_100)*(TP+FN)*(TN_100+FP_100)*(TN_100+FN) + 1e-10)
    mcc_1000  = (TP*TN_1000 - FP_1000*FN) / np.sqrt((TP+FP_1000)*(TP+FN)*(TN_1000+FP_1000)*(TN_1000+FN) + 1e-10)

    if precision_shuffle + recall > 0:
        f1_shuffle = 2 * precision_shuffle * recall / (precision_shuffle + recall)
    else:
        f1_shuffle = 0
    if precision_100 + recall > 0:
        f1_100 = 2 * precision_100 * recall / (precision_100 + recall)
    else:
        f1_100 = 0
    if precision_1000 + recall > 0:
        f1_1000 = 2 * precision_1000 * recall / (precision_1000 + recall)
    else:
        f1_1000 = 0

    y_recalls.append(recall)
    y_precisions_shuffle.append(precision_shuffle)
    y_precisions_100.append(precision_100)
    y_precisions_1000.append(precision_1000)
    y_f1s_shuffle.append(f1_shuffle)
    y_f1s_100.append(f1_100)
    y_f1s_1000.append(f1_1000)
    y_accuracy_shuffle.append(accuracy_shuffle)
    y_accuracy_100.append(accuracy_100)
    y_accuracy_1000.append(accuracy_1000)
    y_mcc_shuffle.append(mcc_shuffle)
    y_mcc_100.append(mcc_100)
    y_mcc_1000.append(mcc_1000)
    


In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(18, 5))

# Recalls
axs[0].plot(thresholds, y_recalls )
axs[0].set_xlabel("Delta")
axs[0].set_ylabel("Recall")
axs[0].set_title("Recalls")
axs[0].legend()

# Precision shuffle
axs[1].plot(thresholds, y_precisions_shuffle )
axs[1].set_xlabel("Delta")
axs[1].set_ylabel("Precision")
axs[1].set_title("Precision (shuffle)")
axs[1].legend()

# Precision 100km
axs[2].plot(thresholds, y_precisions_100)
axs[2].set_xlabel("Delta")
axs[2].set_ylabel("Precision")
axs[2].set_title("Precision (100km)")
axs[2].legend()
# Precision 1000km
axs[3].plot(thresholds, y_precisions_1000)
axs[3].set_xlabel("Delta")
axs[3].set_ylabel("Precision")
axs[3].set_title("Precision (1000km)")
axs[3].legend()

fig.suptitle("Recalls and Precisions given different deltas for the YFCC dataset", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Recalls
axs[0].plot(y_recalls, y_precisions_shuffle )
axs[0].set_xlabel("Recall")
axs[0].set_ylabel("Precision")
axs[0].set_title("Precision-Recall curve (shuffle)")
axs[0].legend()

# Precision 100km
axs[1].plot(y_recalls, y_precisions_100)
axs[1].set_xlabel("Recall")
axs[1].set_ylabel("Precision")
axs[1].set_title("Precision-Recall curve (100km)")
axs[1].legend()
# Precision 1000km
axs[2].plot(y_recalls, y_precisions_1000)
axs[2].set_xlabel("Recall")
axs[2].set_ylabel("Precision")
axs[2].set_title("Precision-Recall curve(1000km)")
axs[2].legend()

fig.suptitle("Precision-Recall curve for the YFCC dataset", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Recalls
axs[0].plot(thresholds, y_f1s_shuffle )
axs[0].axhline(y=2/3, color='red', linestyle='--')
axs[0].set_xlabel("Delta")
axs[0].set_ylabel("F1 score")
axs[0].set_title("F1 score for different deltas (shuffle)")
axs[0].legend()

# Precision 100km
axs[1].plot(thresholds, y_f1s_100)
axs[1].axhline(y=2/3, color='red', linestyle='--')
axs[1].set_xlabel("Delta")
axs[1].set_ylabel("F1 score")
axs[1].set_title("F1 score for different deltas (100km)")
axs[1].legend()
# Precision 1000km
axs[2].plot(thresholds, y_f1s_1000)
axs[2].axhline(y=2/3, color='red', linestyle='--')
axs[2].set_xlabel("Delta")
axs[2].set_ylabel("F1 score")
axs[2].set_title("F1 score for different deltas (1000km)")
axs[2].legend()

fig.suptitle("F1 score given different deltas for the YFCC dataset", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Recalls
axs[0].plot(thresholds, y_accuracy_shuffle )
axs[0].set_xlabel("Delta")
axs[0].set_ylabel("Accuracy")
axs[0].set_title("Accuracy for different deltas (shuffle)")
axs[0].legend()

# Precision 100km
axs[1].plot(thresholds, y_accuracy_100)
axs[1].set_xlabel("Delta")
axs[1].set_ylabel("Accuracy")
axs[1].set_title("Accuracy for different deltas (100km)")
axs[1].legend()
# Precision 1000km
axs[2].plot(thresholds, y_accuracy_1000)
axs[2].set_xlabel("Delta")
axs[2].set_ylabel("Accuracy")
axs[2].set_title("Accuracy for different deltas (1000km)")
axs[2].legend()

fig.suptitle("Accuracy given different deltas for the YFCC dataset", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Recalls
axs[0].plot(thresholds, y_mcc_shuffle )
axs[0].set_xlabel("Delta")
axs[0].set_ylabel("MCC")
axs[0].set_title("MCC for different deltas (shuffle)")
axs[0].legend()

# Precision 100km
axs[1].plot(thresholds, y_mcc_100)
axs[1].set_xlabel("Delta")
axs[1].set_ylabel("MCC")
axs[1].set_title("MCC for different deltas (100km)")
axs[1].legend()
# Precision 1000km
axs[2].plot(thresholds, y_mcc_1000)
axs[2].set_xlabel("Delta")
axs[2].set_ylabel("MCC")
axs[2].set_title("MCC for different deltas (1000km)")
axs[2].legend()

fig.suptitle("Matthews Correlation Coefficient given different deltas for the YFCC dataset", fontsize=16)
plt.tight_layout()
plt.show()

##### With landscapes images (osv dataset)

In [ ]:
thresholds = np.linspace(
    min(o_true_scores.min(), o_false_scores_shuffle.min(),o_false_scores_100.min(),o_false_scores_1000.min() ),
    max(o_true_scores.max(), o_false_scores_shuffle.max(),o_false_scores_100.max(), o_false_scores_1000.max()),
    200
)

o_recalls = []
o_precisions_shuffle = []
o_precisions_100 = []
o_precisions_1000 = []
o_f1s_shuffle = []
o_f1s_100 = []
o_f1s_1000 = []
o_accuracy_shuffle = []
o_accuracy_100 = []
o_accuracy_1000 = []
o_mcc_shuffle = []
o_mcc_100 = []
o_mcc_1000 = []



for t in thresholds:

    TP = (o_true_scores > t).sum().item()
    FN = (o_true_scores <= t).sum().item()

    FP_shuffle = (o_false_scores_shuffle > t).sum().item()
    TN_shuffle = (o_false_scores_shuffle <= t).sum().item()

    FP_100 = (o_false_scores_100 > t).sum().item()
    TN_100 = (o_false_scores_100 <= t).sum().item()

    FP_1000 = (o_false_scores_1000 > t).sum().item()
    TN_1000 = (o_false_scores_1000 <= t).sum().item()


    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    precision_shuffle = TP / (TP + FP_shuffle) if (TP + FP_shuffle) > 0 else 1
    precision_100 = TP / (TP + FP_100) if (TP + FP_100) > 0 else 1
    precision_1000 = TP / (TP + FP_1000) if (TP + FP_1000) > 0 else 1
    accuracy_shuffle = (TP + TN_shuffle) / (TP + FN + FP_shuffle + TN_shuffle)
    accuracy_100 = (TP + TN_100) / (TP + FN + FP_100 + TN_100)
    accuracy_1000 = (TP + TN_1000) / (TP + FN + FP_1000 + TN_1000)
    mcc_shuffle = (TP*TN_shuffle - FP_shuffle*FN) / np.sqrt((TP+FP_shuffle)*(TP+FN)*(TN_shuffle+FP_shuffle)*(TN_shuffle+FN) + 1e-10)
    mcc_100   = (TP*TN_100 - FP_100*FN) / np.sqrt((TP+FP_100)*(TP+FN)*(TN_100+FP_100)*(TN_100+FN) + 1e-10)
    mcc_1000  = (TP*TN_1000 - FP_1000*FN) / np.sqrt((TP+FP_1000)*(TP+FN)*(TN_1000+FP_1000)*(TN_1000+FN) + 1e-10)


    if precision_shuffle + recall > 0:
        f1_shuffle = 2 * precision_shuffle * recall / (precision_shuffle + recall)
    else:
        f1_shuffle = 0
    if precision_100 + recall > 0:
        f1_100 = 2 * precision_100 * recall / (precision_100 + recall)
    else:
        f1_100 = 0
    if precision_1000 + recall > 0:
        f1_1000 = 2 * precision_1000 * recall / (precision_1000 + recall)
    else:
        f1_1000 = 0

    o_recalls.append(recall)
    o_precisions_shuffle.append(precision_shuffle)
    o_precisions_100.append(precision_100)
    o_precisions_1000.append(precision_1000)
    o_f1s_shuffle.append(f1_shuffle)
    o_f1s_100.append(f1_100)
    o_f1s_1000.append(f1_1000)
    o_accuracy_shuffle.append(accuracy_shuffle)
    o_accuracy_100.append(accuracy_100)
    o_accuracy_1000.append(accuracy_1000)
    o_mcc_shuffle.append(mcc_shuffle)
    o_mcc_100.append(mcc_100)
    o_mcc_1000.append(mcc_1000)


In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(18, 5))

# Recalls
axs[0].plot(thresholds, y_recalls )
axs[0].set_xlabel("Delta")
axs[0].set_ylabel("Recall")
axs[0].set_title("Recalls")
axs[0].legend()

# Precision shuffle
axs[1].plot(thresholds, y_precisions_shuffle )
axs[1].set_xlabel("Delta")
axs[1].set_ylabel("Precision")
axs[1].set_title("Precision (shuffle)")
axs[1].legend()

# Precision 100km
axs[2].plot(thresholds, y_precisions_100)
axs[2].set_xlabel("Delta")
axs[2].set_ylabel("Precision")
axs[2].set_title("Precision (100km)")
axs[2].legend()
# Precision 1000km
axs[3].plot(thresholds, y_precisions_1000)
axs[3].set_xlabel("Delta")
axs[3].set_ylabel("Precision")
axs[3].set_title("Precision (1000km)")
axs[3].legend()

fig.suptitle("Recalls and Precisions given different deltas for the YFCC dataset", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Recalls
axs[0].plot(o_recalls, o_precisions_shuffle )
axs[0].set_xlabel("Recall")
axs[0].set_ylabel("Precision")
axs[0].set_title("Precision-Recall curve (shuffle)")
axs[0].legend()

# Precision 100km
axs[1].plot(o_recalls, o_precisions_100)
axs[1].set_xlabel("Recall")
axs[1].set_ylabel("Precision")
axs[1].set_title("Precision-Recall curve (100km)")
axs[1].legend()
# Precision 1000km
axs[2].plot(o_recalls, o_precisions_1000)
axs[2].set_xlabel("Recall")
axs[2].set_ylabel("Precision")
axs[2].set_title("Precision-Recall curve(1000km)")
axs[2].legend()

fig.suptitle("Precision-Recall curve for the OSV dataset", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Recalls
axs[0].plot(thresholds, o_f1s_shuffle )
axs[0].axhline(y=2/3, color='red', linestyle='--')
axs[0].set_xlabel("Delta")
axs[0].set_ylabel("F1 score")
axs[0].set_title("F1 score for different deltas (shuffle)")
axs[0].legend()

# Precision 100km
axs[1].plot(thresholds, o_f1s_100)
axs[1].axhline(y=2/3, color='red', linestyle='--')
axs[1].set_xlabel("Delta")
axs[1].set_ylabel("F1 score")
axs[1].set_title("F1 score for different deltas (100km)")
axs[1].legend()
# Precision 1000km
axs[2].plot(thresholds, o_f1s_1000)
axs[2].axhline(y=2/3, color='red', linestyle='--')
axs[2].set_xlabel("Delta")
axs[2].set_ylabel("F1 score")
axs[2].set_title("F1 score for different deltas (1000km)")
axs[2].legend()

fig.suptitle("F1 score given different deltas for the OSV dataset", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Recalls
axs[0].plot(thresholds, o_accuracy_shuffle )
axs[0].set_xlabel("Delta")
axs[0].set_ylabel("Accuracy")
axs[0].set_title("Accuracy for different deltas (shuffle)")
axs[0].legend()

# Precision 100km
axs[1].plot(thresholds, o_accuracy_100)
axs[1].set_xlabel("Delta")
axs[1].set_ylabel("Accuracy")
axs[1].set_title("Accuracy for different deltas (100km)")
axs[1].legend()
# Precision 1000km
axs[2].plot(thresholds, o_accuracy_1000)
axs[2].set_xlabel("Delta")
axs[2].set_ylabel("Accuracy")
axs[2].set_title("Accuracy for different deltas (1000km)")
axs[2].legend()

fig.suptitle("Accuracy given different deltas for the OSV dataset", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Recalls
axs[0].plot(thresholds, o_mcc_shuffle )
axs[0].set_xlabel("Delta")
axs[0].set_ylabel("MCC")
axs[0].set_title("MCC for different deltas (shuffle)")
axs[0].legend()

# Precision 100km
axs[1].plot(thresholds, o_mcc_100)
axs[1].set_xlabel("Delta")
axs[1].set_ylabel("MCC")
axs[1].set_title("MCC for different deltas (100km)")
axs[1].legend()
# Precision 1000km
axs[2].plot(thresholds, o_mcc_1000)
axs[2].set_xlabel("Delta")
axs[2].set_ylabel("MCC")
axs[2].set_title("MCC for different deltas (1000km)")
axs[2].legend()

fig.suptitle("Matthews Correlation Coefficient given different deltas for the OSV dataset", fontsize=16)
plt.tight_layout()
plt.show()